In [4]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from lightning import pytorch as pl
from rdkit import Chem
from chemprop import data, featurizers, models, nn, utils

# ==========================================
# 1. SETUP, CONFIGURATION & DATA CLEANING
# ==========================================
data_dir = Path.cwd()
smiles_col = "smiles"   
target_col = "activity" 
num_workers = 4         

print("Loading and cleaning datasets...")

# --- PRE-TRAINING DATA ---
# Dataset 1: Has 'SMILE' and 'MIC'
df_1 = pd.read_excel(data_dir / "Dataset_1.xlsx")
df_1 = df_1.rename(columns={"SMILE": "smiles"})
# CLEANING FIX: Strip out '>' and '<' symbols and convert to numeric
df_1["MIC"] = pd.to_numeric(df_1["MIC"].astype(str).str.replace('>', '').str.replace('<', '').str.strip(), errors='coerce')
df_1["activity"] = (df_1["MIC"] <= 16).astype(float)
df_1 = df_1[["smiles", "activity"]]

# Dataset 2: Has no headers. 
df_2 = pd.read_excel(data_dir / "Dataset_2.xlsx", header=None)
df_2 = df_2.rename(columns={1: "smiles", 2: "MIC"})
# CLEANING FIX: Strip out '>' and '<' symbols and convert to numeric
df_2["MIC"] = pd.to_numeric(df_2["MIC"].astype(str).str.replace('>', '').str.replace('<', '').str.strip(), errors='coerce')
df_2["activity"] = (df_2["MIC"] <= 16).astype(float)
df_2 = df_2[["smiles", "activity"]]

# Dataset 3: Has PUBCHEM columns
df_3 = pd.read_excel(data_dir / "Dataset_3.xlsx")
df_3 = df_3.rename(columns={"PUBCHEM_EXT_DATASOURCE_SMILES": "smiles"})
df_3["activity"] = df_3["PUBCHEM_ACTIVITY_OUTCOME"].map({"Active": 1.0, "Inactive": 0.0})
df_3 = df_3[["smiles", "activity"]]

# Combine all pre-training data cleanly and drop any rows with missing values
df_pre = pd.concat([df_1, df_2, df_3], ignore_index=True).dropna()


# --- FINE-TUNING DATA ---
# Dataset 4: Has PUBCHEM columns
df_fine = pd.read_excel(data_dir / "Dataset_4.xlsx")
df_fine = df_fine.rename(columns={"PUBCHEM_EXT_DATASOURCE_SMILES": "smiles"})
df_fine["activity"] = df_fine["PUBCHEM_ACTIVITY_OUTCOME"].map({"Active": 1.0, "Inactive": 0.0})
df_fine = df_fine[["smiles", "activity"]].dropna()


# --- TEST SET DATA ---
# Dataset 5: Has PUBCHEM columns
df_test = pd.read_csv(data_dir / "Dataset_5.csv")
df_test = df_test.rename(columns={"PUBCHEM_EXT_DATASOURCE_SMILES": "smiles"})
df_test["activity"] = df_test["PUBCHEM_ACTIVITY_OUTCOME"].map({"Active": 1.0, "Inactive": 0.0})
df_test = df_test[["smiles", "activity"]].dropna()

print(f"Pre-training set size: {len(df_pre)} molecules")
print(f"Fine-tuning set size: {len(df_fine)} molecules")
print(f"Test set size: {len(df_test)} molecules")

# ==========================================
# 2. HELPER FUNCTION: PREPARE DATALOADERS
# ==========================================
def prepare_data(df, is_train=True):
    smis = df[smiles_col].values
    ys = df[[target_col]].values
    
    # Generate RDKit mols
    mols = [utils.make_mol(smi, keep_h=False, add_h=False) for smi in smis]
    
    # Generate 1D ECFP / Morgan Fingerprints (Extra Molecule Features)
    mol_featurizer = featurizers.MorganBinaryFeaturizer()
    extra_mol_features = np.array([mol_featurizer(mol) for mol in mols])
    
    # Create Datapoints with extra descriptors (x_d)
    datapoints = [
        data.MoleculeDatapoint(mol, y, x_d=x_d)
        for mol, y, x_d in zip(mols, ys, extra_mol_features)
    ]
    
    # Featurize Graph
    graph_featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
    dset = data.MoleculeDataset(datapoints, graph_featurizer)
    
    # Scale the extra 1D descriptors (FIXED: "x_d" to "X_d")
    x_d_scaler = dset.normalize_inputs("X_d") if is_train else None
    
    return dset, x_d_scaler, extra_mol_features.shape[1], graph_featurizer

# ==========================================
# 3. PHASE 1: PRE-TRAINING
# ==========================================
print("--- Starting Pre-training ---")
pre_dset, pre_xd_scaler, n_extra_descs, graph_feat = prepare_data(df_pre, is_train=True)

# Split pre-training data
train_idx, val_idx, test_idx = data.make_split_indices([d.mol for d in pre_dset.data])
pre_train_data, pre_val_data, _ = data.split_data_by_indices(pre_dset.data, train_idx, val_idx, test_idx)

# Create final Datasets and Loaders (FIXED: Added [0] to extract the first replicate)
pre_train_dset = data.MoleculeDataset(pre_train_data[0], graph_feat)
pre_val_dset = data.MoleculeDataset(pre_val_data[0], graph_feat)

# Apply the scaler to the validation set (FIXED: "x_d" to "X_d")
pre_val_dset.normalize_inputs("X_d", pre_xd_scaler)

pre_train_loader = data.build_dataloader(pre_train_dset, num_workers=num_workers)
pre_val_loader = data.build_dataloader(pre_val_dset, num_workers=num_workers, shuffle=False)

# Build the Multimodal Classification Model
X_d_transform = nn.ScaleTransform.from_standard_scaler(pre_xd_scaler)

mp = nn.BondMessagePassing(d_v=graph_feat.atom_fdim, d_e=graph_feat.bond_fdim)
agg = nn.MeanAggregation()
ffn_input_dim = mp.output_dim + n_extra_descs

# Binary Classification FFN and AUROC metric
ffn = nn.BinaryClassificationFFN(input_dim=ffn_input_dim, n_tasks=1)
metric_list = [nn.metrics.BinaryAUROC()]

mpnn = models.MPNN(mp, agg, ffn, batch_norm=False, metrics=metric_list, X_d_transform=X_d_transform)

# Train the pre-trained model
pre_trainer = pl.Trainer(
    logger=False, 
    enable_checkpointing=True, 
    max_epochs=20, 
    accelerator="auto"
)
pre_trainer.fit(mpnn, pre_train_loader, pre_val_loader)

# Save checkpoint path for transfer learning
checkpoint_path = pre_trainer.checkpoint_callback.best_model_path
print(f"Pre-training complete. Model saved to: {checkpoint_path}")

# ==========================================
# 4. PHASE 2: TRANSFER LEARNING / FINE-TUNING
# ==========================================
print("--- Starting Fine-tuning ---")

# Load fine-tuning data
fine_dset, fine_xd_scaler, _, _ = prepare_data(df_fine, is_train=True)
fine_train_idx, fine_val_idx, _ = data.make_split_indices([d.mol for d in fine_dset.data])
fine_train_data, fine_val_data, _ = data.split_data_by_indices(fine_dset.data, fine_train_idx, fine_val_idx, [])

# Create final Datasets and Loaders (FIXED: Added [0] to extract the first replicate)
fine_train_dset = data.MoleculeDataset(fine_train_data[0], graph_feat)
fine_val_dset = data.MoleculeDataset(fine_val_data[0], graph_feat)

# Apply scaler to fine-tuning validation set (FIXED: "x_d" to "X_d")
fine_val_dset.normalize_inputs("X_d", fine_xd_scaler)

fine_train_loader = data.build_dataloader(fine_train_dset, num_workers=num_workers)
fine_val_loader = data.build_dataloader(fine_val_dset, num_workers=num_workers, shuffle=False)

# Load the pre-trained model weights
finetune_mpnn = models.MPNN.load_from_checkpoint(checkpoint_path)

# FREEZE the Message Passing (Graph) layers to retain general structural knowledge
finetune_mpnn.message_passing.apply(lambda module: module.requires_grad_(False))
finetune_mpnn.message_passing.eval()

# Train the fine-tuned model (optimizing only the FFN layers)
fine_trainer = pl.Trainer(
    logger=False, 
    enable_checkpointing=True, 
    max_epochs=20, 
    accelerator="auto"
)
fine_trainer.fit(finetune_mpnn, fine_train_loader, fine_val_loader)

# ==========================================
# 5. PHASE 3: PREDICTION ON TEST SET
# ==========================================
print("--- Evaluating Test Set ---")

# Prepare test data (do not fit scaler, apply the fine-tuning scaler!)
test_dset, _, _, _ = prepare_data(df_test, is_train=False)

# Apply scaler to test set (FIXED: "x_d" to "X_d")
test_dset.normalize_inputs("X_d", fine_xd_scaler)
test_loader = data.build_dataloader(test_dset, num_workers=num_workers, shuffle=False)

# Test the model
results = fine_trainer.test(finetune_mpnn, test_loader)
print("Test Results:", results)

Loading and cleaning datasets...
Pre-training set size: 9937 molecules
Fine-tuning set size: 2809 molecules
Test set size: 892 molecules
--- Starting Pre-training ---


The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA GeForce RTX 4060') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEV

┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │  705 K │ train │     0 │
│ 4 │ X_d_transform   │ ScaleTransform          │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 932 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 932 K                                                                                                
Total estimated model params size (MB): 3                                                                          
Modules in train mode: 24                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/media/biopolymer/1TB/biopolymer/conda_envs/chemprop2/lib/python3.11/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


--- Evaluating Test Set ---


/media/biopolymer/1TB/biopolymer/conda_envs/chemprop2/lib/python3.11/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/roc          │    0.5492709279060364     │
└───────────────────────────┴───────────────────────────┘

Test Results: [{'test/roc': 0.5492709279060364}]
